In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("../telco.duckdb")

econ = con.execute("""
    SELECT s.customer_id,
           s.clv,
           s.value_decile,
           s.churn_probability,
           s.churn_value,
           f.contract,
           f.tenure_band,
           f.monthly_charges
    FROM customer_scores s
    JOIN vw_customer_features f ON s.customer_id = f.customer_id
""").df()

print(econ.shape)
econ.head()

(7043, 8)


,customer_id,clv,value_decile,churn_probability,churn_value,contract,tenure_band,monthly_charges
0,3668-QPYBK,193.86,8,0.551139,1,Month-to-month,New,53.85
1,9237-HQITU,254.52,7,0.838677,1,Month-to-month,New,70.70
2,9305-CDSKC,358.74,4,0.893799,1,Month-to-month,New,99.65
3,7892-POOKP,377.28,4,0.808888,1,Month-to-month,Established,104.80
4,0280-XJGEX,373.32,4,0.591416,1,Month-to-month,Loyal,103.70


In [6]:
con.execute(open("../sql/05_economics.sql").read())

con.execute("""
    SELECT decision,
           COUNT(*) AS customers,
           ROUND(SUM(expected_value_saved),2) AS value_saved,
           ROUND(SUM(intervention_cost),2) AS total_cost,
           ROUND(SUM(net_value),2) AS net
    FROM vw_intervention_economics
    GROUP BY 1
""").df()

,decision,customers,value_saved,total_cost,net
0,Do not intervene,4825,105819.35,289500.0,-183680.65
1,Intervene,2218,172742.37,133080.0,39662.37


In [7]:
con.execute("""
    SELECT ROUND(MIN(expected_value_saved),2)  AS min_saved,
           ROUND(AVG(expected_value_saved),2)  AS avg_saved,
           ROUND(MAX(expected_value_saved),2)  AS max_saved,
           ROUND(QUANTILE_CONT(expected_value_saved, 0.90),2) AS p90,
           ROUND(QUANTILE_CONT(expected_value_saved, 0.99),2) AS p99
    FROM vw_intervention_economics
""").df()

,min_saved,avg_saved,max_saved,p90,p99
0,0.56,39.55,163.09,82.28,109.24


In [8]:
con.execute("""
    SELECT value_decile,
           COUNT(*) AS customers,
           SUM(CASE WHEN decision = 'Intervene' THEN 1 ELSE 0 END) AS intervene,
           ROUND(SUM(CASE WHEN decision = 'Intervene' THEN net_value ELSE 0 END),2) AS net_if_targeted,
           ROUND(SUM(net_value),2) AS net_if_all_contacted
    FROM vw_intervention_economics
    GROUP BY 1 ORDER BY 1
""").df()

,value_decile,customers,intervene,net_if_targeted,net_if_all_contacted
0,1,705,83.0,1599.74,-17760.85
1,2,705,349.0,10261.63,-2598.32
2,3,705,138.0,2642.34,-17693.18
3,4,704,589.0,13506.37,11195.62
4,5,704,532.0,8206.15,5215.91
5,6,704,363.0,2974.06,-9179.05
6,7,704,164.0,472.08,-19150.76
7,8,704,0.0,0.00,-28863.49
8,9,704,0.0,0.00,-30433.87
9,10,704,0.0,0.00,-34750.29
